# Cuaderno U2-02. Entorno de trabajo, ingreso de datos y configuración

**Modelación y Simulación Computacional** · Maestría en Ingeniería, Universidad de Sucre, periodo 2026-2
**Unidad 2.** Herramientas computacionales para modelación y simulación
**Subtema del plan.** 2.2 Entorno de trabajo, ingreso de datos y configuración de modelos
**Autor.** Prof. Daniel Otero Meza, Ing., Ph.D.

<!-- ENLACE_COLAB -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msc-unisucre/msc2026-material/blob/main/03_cuadernos/Unidad2/U2_02_entorno_datos_y_configuracion.ipynb)


Este cuaderno recorre la Sección 2.2 del libro. Trabaja sobre el
archivo de la estación de bombeo que abre el capítulo, con separador de punto
y coma, coma decimal, encabezados con unidad, marcas de tiempo desordenadas,
duplicados, huecos y valores atípicos. Reproduce el Ejemplo 2.1 y el Ejemplo
2.2 con las cifras que el libro publica.

## Objetivos de aprendizaje

Al terminar este cuaderno el estudiante debe ser capaz de lo siguiente.

1. Leer un archivo de instrumentación con tipos, separador, coma decimal y marcas de tiempo declarados, como en el Listado 2.2.
2. Producir el informe cuantitativo de las verificaciones V1 a V6 de la Tabla 2.4 y reproducir el Ejemplo 2.1 del libro.
3. Aplicar el criterio robusto de la Ecuación 2.1 y reproducir las cifras del Ejemplo 2.2 sobre la sonda de humedad.
4. Mostrar numéricamente el punto de ruptura del Teorema 2.1 y justificar por qué la desviación estándar no sirve para detectar.
5. Agregar una serie con cálculo simultáneo de cobertura e invalidar los intervalos que no la alcanzan, como en el Listado 2.5.

## Puesta a punto

La primera celda instala lo que falte y la segunda fija la semilla del curso,
la paleta del libro y la función que compara cada resultado con el valor
publicado. Ningún resultado de este cuaderno depende de una ejecución
concreta.

In [ ]:
# Puesta a punto. Detecta el entorno e instala solo lo que falte.
import importlib
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules


def asegurar(paquetes: dict[str, str]) -> None:
    """Instala los paquetes cuyo módulo no se encuentre en el entorno."""
    faltantes = [p for p, m in paquetes.items()
                 if importlib.util.find_spec(m) is None]
    if faltantes:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        *faltantes], check=True)


asegurar({"numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
          "matplotlib": "matplotlib", "sympy": "sympy"})
print("entorno listo, Colab =", EN_COLAB)

In [ ]:
# Configuración común a todos los cuadernos del curso.
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sympy as sp

SEMILLA = 20262
rng = np.random.default_rng(SEMILLA)

PALETA = {"azul": "#1F4E79", "rojo": "#B3251E", "verde": "#2E7D32",
          "naranja": "#E07B00", "gris": "#5A5A5A", "morado": "#6A3D9A"}

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 110,
                     "font.size": 9, "axes.grid": True,
                     "grid.linewidth": 0.4, "grid.alpha": 0.5,
                     "axes.prop_cycle": plt.cycler(color=list(PALETA.values()))})


def contra_libro(nombre: str, calculado: float, publicado: float,
                 unidad: str = "", tol: float = 1e-3, relativa: bool = True,
                 exigir: bool = True, nota: str = "") -> None:
    """Compara un resultado del cuaderno con el valor que publica el libro.

    Detiene la ejecución si la diferencia excede la tolerancia y `exigir` es
    verdadero. Las magnitudes que dependen de la máquina, como los tiempos de
    ejecución, se informan con `exigir=False` y una nota que lo advierte.
    """
    error = abs(calculado - publicado)
    if relativa and publicado != 0.0:
        error = error / abs(publicado)
    ok = error <= tol
    print(f"{nombre:<44s} cuaderno {calculado:>13.6g}  "
          f"libro {publicado:>13.6g} {unidad:<9s} "
          f"{'coincide' if ok else 'DIFIERE '}{nota}")
    if exigir and not ok:
        raise AssertionError(
            f"{nombre}, el cuaderno da {calculado!r} y el libro publica "
            f"{publicado!r}, con error {error:.3e}")


print("NumPy", np.__version__, "| SciPy", scipy.__version__,
      "| pandas", pd.__version__, "| SymPy", sp.__version__)

### Acceso a los datos

Los archivos viven en `03_cuadernos/datos/`. La función `ruta_datos` intenta
primero la ruta relativa del repositorio y, si el archivo no está, lo regenera
con la semilla del curso. Así el cuaderno corre igual en Colab, donde no
existe la carpeta, y en una instalación local. Nunca se usan rutas absolutas
del computador del docente.

In [ ]:
# Acceso a los datos. Se intenta la ruta relativa del repositorio y, si el
# archivo no existe, se regenera con la semilla del curso.
EXTRACTO_CAUDAL = [
    ("2026-03-04 08:00:00", "41,8", "3,42"),
    ("2026-03-04 08:05:00", "42,3", "3,40"),
    ("2026-03-04 08:10:00", "43,1", "3,38"),
    ("2026-03-04 08:15:00", "-9,9", "3,41"),
    ("2026-03-04 08:20:00", "44,0", "3,36"),
    ("2026-03-04 08:25:00", "", "3,35"),
    ("2026-03-04 08:30:00", "43,6", "3,37"),
    ("2026-03-04 08:35:00", "186,0", "3,33"),
    ("2026-03-04 08:45:00", "44,5", "3,31"),
    ("2026-03-04 08:40:00", "43,9", "3,34"),
    ("2026-03-04 08:50:00", "44,2", "3,30"),
    ("2026-03-04 08:50:00", "43,9", "3,30"),
]


def _gen_caudal_bombeo(destino: Path) -> None:
    """Extracto sucio de doce registros del Ejemplo 2.1 del libro."""
    lineas = ["fecha_hora;caudal_l_s;presion_bar"]
    lineas += [";".join(fila) for fila in EXTRACTO_CAUDAL]
    destino.write_text("\n".join(lineas) + "\n", encoding="utf-8")


def _gen_estacion_bombeo(destino: Path) -> None:
    """Tres días de operación de la estación de bombeo, cada 5 min."""
    generador = np.random.default_rng(SEMILLA)
    t = pd.date_range("2026-03-04 00:00", periods=864, freq="5min")
    h = t.hour + t.minute / 60.0
    consigna = np.where((h >= 5.0) & (h < 11.0), 44.0,
                        np.where((h >= 15.0) & (h < 20.0), 47.5, 12.0))
    q = consigna + 0.9 * generador.standard_normal(t.size)
    p = 3.35 + 0.018 * (q - 40.0) + 0.03 * generador.standard_normal(t.size)
    df = pd.DataFrame({"fecha_hora": t, "caudal_l_s": q, "presion_bar": p})
    for inicio, largo in ((150, 38), (402, 19), (640, 7), (806, 12)):
        df.loc[inicio:inicio + largo - 1,
               ["caudal_l_s", "presion_bar"]] = np.nan
    df.loc[233, "caudal_l_s"] = -11.4
    df.loc[234, "caudal_l_s"] = -8.7
    df.loc[521, "caudal_l_s"] = 186.0
    df.loc[522, "caudal_l_s"] = 141.2
    for i in (318, 319, 320, 705):
        df.loc[i, "caudal_l_s"] = df.loc[i, "caudal_l_s"] + 22.0
    df = pd.concat([df, df.loc[[300, 301, 302, 588, 589]].copy()],
                   ignore_index=True)
    orden = list(range(len(df)))
    orden[300:306] = orden[303:306] + orden[300:303]
    df = df.iloc[orden].reset_index(drop=True)
    df["fecha_hora"] = df["fecha_hora"].dt.strftime("%Y-%m-%d %H:%M:%S")
    df["caudal_l_s"] = df["caudal_l_s"].round(2)
    df["presion_bar"] = df["presion_bar"].round(3)
    df.to_csv(destino, sep=";", decimal=",", index=False, encoding="utf-8")


def serie_humedad(semilla: int = SEMILLA, n: int = 240, n_cont: int = 24):
    """Sonda de capacitancia cada 30 min durante cinco días, Ejemplo 2.2."""
    generador = np.random.default_rng(semilla)
    t = np.arange(n) * 0.5
    theta = (0.320 - 0.035 * (1.0 - np.exp(-t / 60.0))
             + 0.004 * generador.standard_normal(n))
    idx = np.sort(generador.choice(n, size=n_cont, replace=False))
    contaminada = theta.copy()
    contaminada[idx] += 0.10
    return t, theta, contaminada, idx


def _gen_humedad_suelo(destino: Path) -> None:
    t, _, contaminada, _ = serie_humedad()
    marca = pd.date_range("2026-03-02 00:00", periods=t.size, freq="30min")
    pd.DataFrame({"fecha_hora": marca.strftime("%Y-%m-%d %H:%M:%S"),
                  "humedad_m3_m3": np.round(contaminada, 6)}).to_csv(
        destino, index=False, encoding="utf-8")


GENERADORES = {
    "caudal_bombeo.csv": _gen_caudal_bombeo,
    "estacion_bombeo.csv": _gen_estacion_bombeo,
    "humedad_suelo.csv": _gen_humedad_suelo,
}

CANDIDATAS = [Path("datos"), Path("..") / "datos",
              Path("03_cuadernos") / "datos", Path("..") / ".." / "datos"]


def ruta_datos(nombre: str) -> Path:
    """Devuelve la ruta del archivo de datos, generándolo si hace falta."""
    for base in CANDIDATAS:
        candidata = base / nombre
        if candidata.is_file():
            return candidata
    generada = Path("salida") / "datos_generados"
    generada.mkdir(parents=True, exist_ok=True)
    destino = generada / nombre
    if not destino.is_file():
        GENERADORES[nombre](destino)
    return destino

print("datos disponibles en", ruta_datos("caudal_bombeo.csv").parent)

## 1. La lectura declarada

El libro insiste en que nada se deje al azar al leer un archivo de
instrumentación. La primera celda lee el archivo con los valores por omisión
para que se vea el daño, y la segunda reproduce el Listado 2.2 con el
separador, la coma decimal, la codificación, los tipos y la marca de tiempo
declarados. Obsérvese que la lectura descuidada no levanta ningún error, que
es precisamente el problema.

In [ ]:
RUTA = ruta_datos("caudal_bombeo.csv")

descuidado = pd.read_csv(RUTA)
print("forma con valores por omisión:", descuidado.shape)
print(descuidado.dtypes.to_string())
print()
print(descuidado.head(3).to_string(index=False))
print("\nuna sola columna de texto, ningún error y ninguna advertencia")

In [ ]:
# Listado 2.2 del libro, lectura tipada con unidades declaradas.
tipos = {"caudal_l_s": "float64", "presion_bar": "float64"}
crudo = pd.read_csv(RUTA, sep=";", decimal=",", encoding="utf-8",
                    dtype=tipos, parse_dates=["fecha_hora"])
crudo = crudo.rename(columns={"caudal_l_s": "Q", "presion_bar": "p"})
UNIDADES = {"Q": "L/s", "p": "bar"}
crudo["Q"] = crudo["Q"] / 1000.0
UNIDADES["Q"] = "m3/s"

print(crudo.dtypes.to_string())
print()
print(crudo.to_string(index=False))
print("\nunidades declaradas:", UNIDADES)

## 2. Datos ordenados

La Definición 2.3 del libro fija la forma canónica, en la cual cada variable
ocupa una columna, cada observación ocupa una fila y cada unidad de
observación vive en una tabla distinta. Una tabla ancha, con una columna por
mes o una columna que a veces guarda caudal y a veces presión, obliga a
escribir código que muere con ese archivo. La celda siguiente construye la
forma ancha y la convierte a la forma ordenada.

In [ ]:
ancha = pd.DataFrame(
    {"estacion": ["E1", "E2"],
     "2026-01": [41.2, 38.9], "2026-02": [43.7, 39.4],
     "2026-03": [44.1, 40.2]})
print("forma ancha, una columna por mes")
print(ancha.to_string(index=False))

ordenada = ancha.melt(id_vars="estacion", var_name="mes",
                      value_name="caudal_l_s")
ordenada["mes"] = pd.to_datetime(ordenada["mes"], format="%Y-%m")
ordenada = ordenada.sort_values(["estacion", "mes"]).reset_index(drop=True)

print("\nforma ordenada, una variable por columna y una observación por fila")
print(ordenada.to_string(index=False))
assert set(ordenada.columns) == {"estacion", "mes", "caudal_l_s"}
assert len(ordenada) == ancha.shape[0] * (ancha.shape[1] - 1)

## 3. Las seis verificaciones de ingreso

La Tabla 2.4 del libro enumera las seis comprobaciones que se ejecutan sobre
la tabla recién leída y antes de cualquier cálculo. Su propósito no es limpiar
sino detectar, y por eso devuelven un informe cuantitativo. El Listado 2.3 del
libro implementa las tres primeras.

In [ ]:
VERIFICACIONES = pd.DataFrame(
    [("V1", "tipo y unidad", "columna numérica leída como texto",
      "redeclarar tipos"),
     ("V2", "orden temporal", "marcas repetidas o fuera de secuencia",
      "ordenar y eliminar duplicados"),
     ("V3", "rango físico", "valores negativos o imposibles",
      "marcar como ausentes"),
     ("V4", "datos faltantes", "huecos agrupados en el tiempo",
      "documentar el mecanismo"),
     ("V5", "valores atípicos", "saltos que exceden el criterio robusto",
      "marcar sin borrar"),
     ("V6", "cobertura", "pocas muestras por intervalo agregado",
      "rechazar el agregado")],
    columns=["Clave", "Qué comprueba", "Síntoma de falla", "Acción"])
print(VERIFICACIONES.to_string(index=False))
print("\nninguna acción modifica el archivo original")

In [ ]:
# Listado 2.3 del libro.
def verificar(df: pd.DataFrame, tiempo: str,
              limites: dict[str, tuple[float, float]]) -> pd.Series:
    """Devuelve el conteo de anomalías de ingreso por tipo."""
    d = {"filas": len(df),
         "marcas duplicadas": int(df[tiempo].duplicated().sum()),
         "tiempo no monótono": int(not df[tiempo].is_monotonic_increasing)}
    for col, (lo, hi) in limites.items():
        v = df[col]
        d[f"{col} faltante"] = int(v.isna().sum())
        d[f"{col} fuera de rango"] = int((~v.between(lo, hi)
                                          & v.notna()).sum())
    return pd.Series(d)


limites = {"Q": (0.0, 0.120), "p": (0.5, 6.0)}
informe = verificar(crudo, "fecha_hora", limites)
print(informe.to_string())

## 4. Ejemplo 2.1 del libro reproducido

El Ejemplo 2.1 aplica las verificaciones V1 a V3 al extracto de doce registros
de la estación de bombeo y mide el efecto de ignorarlas. El libro publica doce
filas, una marca duplicada, una secuencia no monótona, una lectura ausente y
dos lecturas fuera del intervalo físico, que son menos 9.9 L/s y 186.0 L/s.
Publica además una media válida de 43.478 L/s frente a una media cruda de
51.582 L/s, con un sesgo del 18.6 por ciento hacia arriba.

In [ ]:
fuera = crudo.loc[~crudo["Q"].between(0.0, 0.120) & crudo["Q"].notna(), "Q"]
valida = crudo["Q"].where(crudo["Q"].between(0.0, 0.120))

media_valida = float(valida.mean()) * 1000.0
media_cruda = float(crudo["Q"].mean()) * 1000.0
sesgo = 100.0 * (media_cruda - media_valida) / media_valida

print("lecturas fuera del intervalo físico, en L/s:",
      np.round(fuera.to_numpy() * 1000.0, 1).tolist())
print(f"lecturas válidas {int(valida.notna().sum())} de {len(crudo)}")
print()
contra_libro("filas del extracto", informe["filas"], 12)
contra_libro("marcas duplicadas", informe["marcas duplicadas"], 1)
contra_libro("tiempo no monótono", informe["tiempo no monótono"], 1)
contra_libro("lecturas ausentes de Q", informe["Q faltante"], 1)
contra_libro("lecturas fuera de rango de Q", informe["Q fuera de rango"], 2)
contra_libro("media de las lecturas válidas", media_valida, 43.478, "L/s",
             tol=2e-5)
contra_libro("media de la columna cruda", media_cruda, 51.582, "L/s",
             tol=2e-5)
contra_libro("sesgo por ignorar las anomalías", sesgo, 18.6, "%", tol=0.05,
             relativa=False)

El comentario del Ejemplo 2.1 señala que la media cruda
sugiere que la bomba opera por encima de su caudal nominal de 45 L/s, que es
la conclusión contraria a la verdadera. La celda siguiente lo pone en
números.

In [ ]:
Q_NOMINAL = 45.0        # L/s
for nombre, valor in (("con las anomalías", media_cruda),
                      ("sin las anomalías", media_valida)):
    veredicto = "por encima" if valor > Q_NOMINAL else "por debajo"
    print(f"{nombre:<20s} {valor:7.3f} L/s, {veredicto} del nominal "
          f"de {Q_NOMINAL:.0f} L/s")
print("\nla marca duplicada, además, duplicaría el peso de ese instante en "
      "cualquier agregación posterior, y ninguna de las tres anomalías "
      "levanta un error al leer el archivo")

## 5. El Algoritmo 2.1 sobre el archivo de tres días

El extracto de doce filas cabe en la pantalla. El archivo real no. La celda
siguiente aplica el Algoritmo 2.1 del libro a tres días de operación de la
misma estación, registrados cada cinco minutos, y devuelve la tabla ordenada
junto con el registro de decisiones. El archivo crudo no se corrige nunca, de
modo que todo descarte queda escrito en el código y no en la memoria de quien
lo hizo.

In [ ]:
RUTA_LARGA = ruta_datos("estacion_bombeo.csv")
largo = pd.read_csv(RUTA_LARGA, sep=";", decimal=",", encoding="utf-8",
                    dtype=tipos, parse_dates=["fecha_hora"])
largo = largo.rename(columns={"caudal_l_s": "Q", "presion_bar": "p"})
largo["Q"] = largo["Q"] / 1000.0

informe_largo = verificar(largo, "fecha_hora", limites)
print(informe_largo.to_string())
print("\nprimeras filas del archivo tal como llegó")
print(largo.head(4).to_string(index=False))

In [ ]:
def preparar(df: pd.DataFrame, tiempo: str,
             limites: dict[str, tuple[float, float]],
             paso: str = "1h", cobertura_min: float = 0.80):
    """Algoritmo 2.1 del libro, con el registro de decisiones."""
    registro = {"informe de ingreso": verificar(df, tiempo, limites)}

    t = df.drop_duplicates(tiempo).sort_values(tiempo).set_index(tiempo)
    registro["filas duplicadas eliminadas"] = len(df) - len(t)

    for col, (lo, hi) in limites.items():
        fuera_rango = (~t[col].between(lo, hi)) & t[col].notna()
        registro[f"{col} marcado por rango"] = int(fuera_rango.sum())
        t[col] = t[col].where(~fuera_rango)

    agregado = t.resample(paso).agg(["mean", "count"])
    esperadas = t.resample(paso).size()
    for col in limites:
        cob = agregado[(col, "count")] / esperadas
        agregado[(col, "cobertura")] = cob
        agregado[(col, "mean")] = agregado[(col, "mean")].where(
            cob >= cobertura_min)
        registro[f"{col} agregados descartados"] = int(
            (cob < cobertura_min).sum())
    return t, agregado.sort_index(axis=1), registro


ordenada_larga, horaria, registro = preparar(largo, "fecha_hora", limites)
for clave, valor in registro.items():
    if isinstance(valor, pd.Series):
        continue
    print(f"  {clave:<32s} {valor}")
print(f"\nfilas ordenadas {len(ordenada_larga)}, intervalos horarios "
      f"{len(horaria)}")
print(horaria["Q"].head(6).round(5).to_string())

## 6. Datos faltantes y su mecanismo

El libro advierte que el origen del hueco determina qué se puede hacer con él.
Un hueco por falla de energía no guarda relación con el valor que se habría
medido, mientras que uno por saturación del sensor ocurre precisamente cuando
la variable era alta. La regla de la asignatura es no imputar antes de
caracterizar el mecanismo, y nunca imputar con la media, que reduce
artificialmente la varianza. La celda siguiente mide los huecos y cuantifica
el daño de imputar con la media.

In [ ]:
def huecos(serie: pd.Series) -> pd.DataFrame:
    """Longitud e inicio de cada racha de valores ausentes."""
    nulo = serie.isna().to_numpy()
    filas, inicio = [], None
    for i, falta in enumerate(nulo):
        if falta and inicio is None:
            inicio = i
        elif not falta and inicio is not None:
            filas.append((serie.index[inicio], i - inicio))
            inicio = None
    if inicio is not None:
        filas.append((serie.index[inicio], len(nulo) - inicio))
    return pd.DataFrame(filas, columns=["inicio", "muestras"])


tabla_huecos = huecos(ordenada_larga["Q"])
print(tabla_huecos.to_string(index=False))
mayor = int(tabla_huecos["muestras"].max())
print(f"\nhueco más largo {mayor} muestras, esto es "
      f"{mayor * 5 / 60:.1f} h de registro perdido")

s = ordenada_larga["Q"].dropna()
imputada = ordenada_larga["Q"].fillna(ordenada_larga["Q"].mean())
print(f"desviación estándar sin imputar {s.std(ddof=1) * 1000:.4f} L/s")
print(f"desviación estándar imputando con la media "
      f"{imputada.std(ddof=1) * 1000:.4f} L/s")
print("imputar con la media reduce la varianza sin agregar información")

## 7. Valores atípicos con criterio robusto

La Definición 2.4 del libro traslada la dificultad a la elección del centro y
de la escala. El criterio clásico usa la media y la desviación estándar, y
falla precisamente cuando más se le necesita, porque ambas se contaminan con
los mismos valores que se pretende detectar. El Listado 2.4 implementa el
puntaje robusto de la Ecuación 2.1, con la escala normalizada mediante el
factor 1.4826.

In [ ]:
# Listado 2.4 del libro.
def atipicos_mad(x, umbral: float = 3.5):
    """Marca como atípico todo dato cuyo puntaje robusto excede el umbral."""
    x = np.asarray(x, dtype=float)
    mediana = np.median(x)
    mad = np.median(np.abs(x - mediana))
    if mad == 0.0:                       # más de la mitad son idénticos
        return np.zeros(x.shape, bool), np.zeros(x.shape), 0.0
    z = 0.6745 * (x - mediana) / mad     # escala robusta = 1.4826 * mad
    return np.abs(z) > umbral, z, 1.4826 * mad


prueba = np.array([1.0, 1.1, 0.9, 1.05, 0.95, 12.0])
marca, z, escala = atipicos_mad(prueba)
print("puntajes robustos:", np.round(z, 2))
print("marcados:", prueba[marca], " escala robusta:", round(escala, 4))

### Ejemplo 2.2 del libro reproducido

Una sonda de capacitancia a 30 cm de profundidad registra la humedad
volumétrica cada 30 min durante cinco días, para un total de 240 lecturas, y
un contacto defectuoso produce 24 lecturas espurias elevadas en
0.10 m3/m3. El libro publica una mediana de 0.29943 m3/m3, una escala robusta
normalizada de 0.01194 m3/m3 y una desviación estándar muestral de
0.03119 m3/m3. El criterio robusto marca las 24 espurias y ninguna válida,
mientras que el clásico marca solamente 8 de las 24 porque su banda, de
semiancho 0.0936 m3/m3, cubre la mayoría de los picos.

In [ ]:
humedad = pd.read_csv(ruta_datos("humedad_suelo.csv"),
                      parse_dates=["fecha_hora"])
theta = humedad["humedad_m3_m3"].to_numpy()
horas = (humedad["fecha_hora"] - humedad["fecha_hora"].iloc[0]
         ).dt.total_seconds().to_numpy() / 3600.0

mediana = float(np.median(theta))
marca_rob, z_rob, escala_rob = atipicos_mad(theta, umbral=3.5)
media_t, desv_t = float(theta.mean()), float(theta.std(ddof=1))
marca_cls = np.abs(theta - media_t) > 3.0 * desv_t

contra_libro("mediana de la serie contaminada", mediana, 0.29943, "m3/m3",
             tol=2e-5)
contra_libro("escala robusta normalizada", escala_rob, 0.01194, "m3/m3",
             tol=5e-4)
contra_libro("desviación estándar muestral", desv_t, 0.03119, "m3/m3",
             tol=2e-4)
contra_libro("semiancho de la banda clásica", 3.0 * desv_t, 0.0936, "m3/m3",
             tol=5e-4)
contra_libro("lecturas marcadas por el criterio robusto",
             int(marca_rob.sum()), 24)
contra_libro("lecturas marcadas por el criterio clásico",
             int(marca_cls.sum()), 8)

In [ ]:
# Panel izquierdo de la Figura 2.5 del libro.
fig, ax = plt.subplots(figsize=(13.5 / 2.54, 6.4 / 2.54), layout="constrained")
ax.plot(horas, theta, ".", ms=3.4, color=PALETA["gris"], label="lectura")
ax.plot(horas[marca_rob], theta[marca_rob], "o", ms=4.6, mfc="none", mew=1.0,
        color=PALETA["rojo"], label="criterio robusto")
ax.plot(horas[marca_cls], theta[marca_cls], "s", ms=7.0, mfc="none", mew=0.9,
        color=PALETA["azul"], label="criterio clásico")
ax.axhline(mediana + 3.5 * escala_rob, color=PALETA["rojo"], lw=0.9, ls="--")
ax.axhline(mediana - 3.5 * escala_rob, color=PALETA["rojo"], lw=0.9, ls="--")
ax.axhline(media_t + 3.0 * desv_t, color=PALETA["azul"], lw=0.9, ls=":")
ax.axhline(media_t - 3.0 * desv_t, color=PALETA["azul"], lw=0.9, ls=":")
ax.set_xlabel("Tiempo desde el inicio del registro (h)")
ax.set_ylabel("Humedad volumétrica (m3/m3)")
ax.set_xlim(0.0, horas[-1])
ax.set_ylim(0.245, 0.505)
ax.legend(loc="upper right", ncols=3, fontsize=7.5)
plt.show()

La verificación del Ejemplo 2.2 aplica el Algoritmo 2.2
sobre el residuo de una mediana móvil de ancho 11 y obtiene una escala robusta
de 0.00511 m3/m3, mucho más cercana al ruido real del instrumento, que es de
0.004 m3/m3. Esa comparación confirma que el valor de 0.01194 m3/m3 estaba
inflado por la tendencia de secado y no por las lecturas espurias.

In [ ]:
def atipicos_con_tendencia(x, ventana: int = 11, umbral: float = 3.5):
    """Algoritmo 2.2 del libro, criterio robusto sobre el residuo."""
    serie = pd.Series(np.asarray(x, dtype=float))
    base = serie.rolling(ventana, center=True, min_periods=1).median()
    residuo = (serie - base).to_numpy()
    return atipicos_mad(residuo, umbral=umbral)


marca_tend, _, escala_tend = atipicos_con_tendencia(theta, ventana=11)
contra_libro("escala robusta sobre el residuo", escala_tend, 0.00511,
             "m3/m3", tol=2e-3)
print(f"ruido declarado del instrumento 0.004 m3/m3, "
      f"escala sin tendencia {escala_tend:.5f} m3/m3")
print(f"lecturas marcadas sobre el residuo {int(marca_tend.sum())}")

## 8. El punto de ruptura del Teorema 2.1

El Teorema 2.1 del libro afirma que el punto de ruptura de la media y de la
desviación estándar vale uno sobre ene, mientras que el de la mediana y el de
la desviación absoluta mediana vale la mitad. La celda siguiente reproduce el
panel derecho de la Figura 2.5, con un contaminante de magnitud un millón, y
comprueba las dos afirmaciones numéricamente.

In [ ]:
def escala_robusta(x) -> float:
    x = np.asarray(x, dtype=float)
    return float(1.4826 * np.median(np.abs(x - np.median(x))))


base = np.random.default_rng(SEMILLA).standard_normal(4000)
MAGNITUD = 1.0e6
fracciones = np.linspace(0.0, 0.62, 125)
s_clasica, s_robusta = [], []
for eps in fracciones:
    z = base.copy()
    k = int(round(eps * z.size))
    if k:
        z[:k] = MAGNITUD
    s_clasica.append(float(z.std(ddof=1)))
    s_robusta.append(escala_robusta(z))
s_clasica = np.array(s_clasica)
s_robusta = np.array(s_robusta)

fig, ax = plt.subplots(figsize=(11.0 / 2.54, 6.4 / 2.54), layout="constrained")
ax.semilogy(fracciones, s_clasica, color=PALETA["azul"], lw=1.4,
            label="desviación estándar")
ax.semilogy(fracciones, s_robusta, color=PALETA["rojo"], lw=1.4,
            label="escala robusta")
ax.axvline(0.5, color=PALETA["gris"], lw=0.9, ls="--")
ax.set_xlabel("Fracción contaminada")
ax.set_ylabel("Escala estimada")
ax.set_xlim(0.0, 0.62)
ax.legend(loc="lower right")
plt.show()

resumen = pd.DataFrame(
    {"fraccion": fracciones, "clasica": s_clasica, "robusta": s_robusta})
muestra = resumen.iloc[[int(np.argmin(np.abs(fracciones - f)))
                        for f in (0.0, 0.005, 0.10, 0.30, 0.45, 0.50, 0.55)]]
print(muestra.to_string(index=False,
                        formatters={"fraccion": "{:.3f}".format,
                                    "clasica": "{:.4g}".format,
                                    "robusta": "{:.4g}".format}))
print("\ncon una sola fracción contaminada del medio por ciento la "
      "desviación estándar ya se fue a diez a la cuatro, mientras que la "
      "escala robusta apenas se mueve. La escala robusta solo se rompe al "
      "alcanzar la mitad de la muestra, que es el valor que el Teorema 2.1 "
      "declara como cota máxima de cualquier estimador equivariante")
assert s_robusta[fracciones < 0.49].max() < 10.0, \
    "la escala robusta no debe dispararse por debajo del punto de ruptura"
assert s_clasica[1] > 1.0e4, "la desviación estándar se rompe con un solo dato"
i50 = int(np.argmin(np.abs(fracciones - 0.5)))
assert s_robusta[i50] > 1.0e4, "en el punto de ruptura la escala robusta cede"

## 9. Agregación y cobertura

El libro advierte que la agregación puede introducir un error mayor que todos
los anteriores, porque un promedio diario calculado sobre cuatro lecturas de
las cuarenta y ocho esperadas no es un promedio diario, es un número que se le
parece. El Listado 2.5 agrega la serie horaria y calcula la cobertura en la
misma operación, y la última línea invalida los agregados con cobertura
insuficiente en lugar de calcularlos de todos modos.

In [ ]:
# Listado 2.5 del libro, sobre la serie de tres días.
serie = largo.drop_duplicates("fecha_hora").sort_values("fecha_hora")
valida_larga = serie.set_index("fecha_hora")["Q"].where(lambda q:
                                                        q.between(0.0, 0.120))
horaria_lst = valida_larga.resample("1h").agg(media="mean", n="count")
horaria_lst["cobertura"] = horaria_lst["n"] / valida_larga.resample("1h").size()
horaria_filtrada = horaria_lst.where(horaria_lst["cobertura"] >= 0.80)

descartados = int((horaria_lst["cobertura"] < 0.80).sum())
print(f"intervalos horarios {len(horaria_lst)}, descartados por cobertura "
      f"{descartados}")
print(horaria_lst.loc[horaria_lst["cobertura"] < 0.80].round(4).to_string())
print("\nmedia diaria con todos los intervalos      "
      f"{horaria_lst['media'].mean() * 1000:.4f} L/s")
print("media diaria solo con cobertura suficiente "
      f"{horaria_filtrada['media'].mean() * 1000:.4f} L/s")

## 10. Ejercicios guiados

Seis celdas incompletas, cada una con su verificación inmediatamente después.
El cuaderno sigue ejecutándose aunque no se completen, porque cada
verificación queda desactivada con su bandera `REVISAR`.

### Ejercicio 1. Lectura declarada

Complete la llamada que lee el archivo grande con el separador, la coma
decimal, la codificación, los tipos y la marca de tiempo declarados.

In [ ]:
# COMPLETE: lea RUTA_LARGA declarando sep, decimal, encoding, dtype y
# parse_dates, de modo que las dos columnas queden como float64 y la marca
# de tiempo como datetime64.
REVISAR_1 = False
leido_ej = pd.read_csv(RUTA_LARGA)          # <- lectura descuidada

In [ ]:
if REVISAR_1:
    assert leido_ej.shape[1] == 3, f"se esperaban 3 columnas y hay {leido_ej.shape[1]}"
    assert str(leido_ej["caudal_l_s"].dtype) == "float64", "el caudal quedó como texto"
    assert str(leido_ej["presion_bar"].dtype) == "float64", "la presión quedó como texto"
    assert leido_ej["fecha_hora"].dtype.kind == "M", "la marca de tiempo no se interpretó"
    print("ejercicio 1 correcto, la lectura queda declarada")
else:
    print("ejercicio 1 pendiente, complete la celda y ponga REVISAR_1 = True")

### Ejercicio 2. Problema 2-20, hueco más largo

Amplíe la función `verificar` del Listado 2.3 para que informe además el hueco
más largo de cada variable, medido en número de muestras consecutivas
ausentes. Este es el Problema 2-20 del libro.

In [ ]:
# COMPLETE: agregue al informe la clave "<col> hueco más largo" con el número
# de muestras consecutivas ausentes de la racha más larga de cada variable.
REVISAR_2 = False


def verificar_ampliada(df, tiempo, limites):
    """Informe del Listado 2.3 más el hueco más largo por variable."""
    d = verificar(df, tiempo, limites).to_dict()
    for col in limites:
        d[f"{col} hueco más largo"] = 0     # <- reemplace por el cálculo
    return pd.Series(d)

In [ ]:
if REVISAR_2:
    ampliado = verificar_ampliada(largo, "fecha_hora", limites)
    print(ampliado.to_string())
    assert ampliado["Q hueco más largo"] == 38, \
        f"el hueco más largo de Q es de 38 muestras, no de {ampliado['Q hueco más largo']}"
    assert ampliado["p hueco más largo"] == 38, "la presión comparte los mismos huecos"
    corta = pd.DataFrame({"t": pd.date_range("2026-01-01", periods=6, freq="h"),
                          "Q": [0.01, np.nan, np.nan, 0.02, np.nan, 0.03]})
    assert verificar_ampliada(corta, "t", {"Q": (0.0, 0.12)})["Q hueco más largo"] == 2, \
        "la racha más larga del caso de prueba es de dos muestras"
    print("\nejercicio 2 correcto, el informe reporta el hueco más largo")
else:
    print("ejercicio 2 pendiente, complete la celda y ponga REVISAR_2 = True")

### Ejercicio 3. Puntaje robusto

Escriba el puntaje robusto de la Ecuación 2.1 del libro sin usar
`atipicos_mad`, y compruebe que coincide con esa función.

In [ ]:
# COMPLETE: devuelva el arreglo de puntajes z de la Ecuación 2.1, esto es
# 0.6745 veces la desviación respecto de la mediana dividida entre la MAD.
REVISAR_3 = False


def puntaje_robusto(x) -> np.ndarray:
    """Puntaje robusto de cada observación."""
    return np.zeros(np.size(x))     # <- reemplace por la Ecuación 2.1

In [ ]:
if REVISAR_3:
    _, z_ref, _ = atipicos_mad(theta)
    z_ej = puntaje_robusto(theta)
    assert np.allclose(z_ej, z_ref), "el puntaje no coincide con el del Listado 2.4"
    assert np.all(puntaje_robusto(np.full(9, 3.0)) == 0.0), \
        "con MAD nula el puntaje debe ser cero y no infinito"
    marcados = int((np.abs(z_ej) > 3.5).sum())
    assert marcados == 24, f"deben marcarse 24 lecturas y se marcaron {marcados}"
    print("ejercicio 3 correcto, el puntaje reproduce la Ecuación 2.1")
else:
    print("ejercicio 3 pendiente, complete la celda y ponga REVISAR_3 = True")

### Ejercicio 4. Cobertura de un agregado

Escriba la función que agrega una serie al paso indicado y devuelve la media
solo cuando la cobertura alcanza el umbral, tal como hace la última línea del
Listado 2.5.

In [ ]:
# COMPLETE: devuelva una tabla con las columnas media, n y cobertura, donde la
# media queda ausente si la cobertura es menor que el umbral.
REVISAR_4 = False


def agregar_con_cobertura(serie, paso="1h", umbral=0.80):
    """Agrega la serie e invalida los intervalos mal cubiertos."""
    tabla = serie.resample(paso).agg(media="mean", n="count")
    tabla["cobertura"] = 1.0       # <- reemplace por la cobertura real
    return tabla

In [ ]:
if REVISAR_4:
    tabla_ej = agregar_con_cobertura(valida_larga)
    assert set(tabla_ej.columns) == {"media", "n", "cobertura"}
    assert np.isclose(tabla_ej["cobertura"].max(), 1.0), "la cobertura máxima es la unidad"
    invalidados = int(tabla_ej["media"].isna().sum())
    assert invalidados == 8, f"deben invalidarse 8 intervalos y se invalidaron {invalidados}"
    peor = tabla_ej["cobertura"].min()
    print(f"cobertura mínima {peor:.3f}, intervalos invalidados {invalidados}")
    print("ejercicio 4 correcto, la cobertura decide qué agregado se reporta")
else:
    print("ejercicio 4 pendiente, complete la celda y ponga REVISAR_4 = True")

### Ejercicio 5. Efecto de las anomalías sobre la media

Calcule la media de la columna cruda y la media de las lecturas que sobreviven
al filtro físico, ambas en litros por segundo, y el sesgo relativo entre
ellas. Los tres valores deben coincidir con los del Ejemplo 2.1.

In [ ]:
# COMPLETE: calcule las dos medias en L/s y el sesgo relativo en por ciento
# sobre la tabla `crudo`, cuya columna Q está en m3/s.
REVISAR_5 = False
media_cruda_ej = 0.0        # <- reemplace por la media de la columna cruda
media_valida_ej = 0.0       # <- reemplace por la media de las válidas
sesgo_ej = 0.0              # <- reemplace por el sesgo relativo en %

In [ ]:
if REVISAR_5:
    contra_libro("media cruda del ejercicio", media_cruda_ej, 51.582, "L/s",
                 tol=2e-5)
    contra_libro("media válida del ejercicio", media_valida_ej, 43.478, "L/s",
                 tol=2e-5)
    contra_libro("sesgo del ejercicio", sesgo_ej, 18.6, "%", tol=0.05,
                 relativa=False)
    print("ejercicio 5 correcto, coincide con el Ejemplo 2.1 del libro")
else:
    print("ejercicio 5 pendiente, complete la celda y ponga REVISAR_5 = True")

### Ejercicio 6. Problema 2-21, efecto de la ventana

Aplique el criterio robusto sobre el residuo de una mediana móvil con la
ventana como argumento y estudie las detecciones al variarla entre 5 y 51.
Este es el Problema 2-21 del libro.

In [ ]:
# COMPLETE: para cada ventana impar entre 5 y 51, cuente cuántas de las
# lecturas marcadas son espurias y cuántas son falsas alarmas. Las espurias
# son las que superan a la serie limpia en más de 0.05 m3/m3.
REVISAR_6 = False
_, limpia, contaminada, indices_espurios = serie_humedad()
ventanas = np.arange(5, 52, 2)
detecciones = pd.DataFrame({"ventana": ventanas, "espurias": 0,
                            "falsas": 0, "escala": np.nan})

In [ ]:
if REVISAR_6:
    print(detecciones.round(5).to_string(index=False))
    assert len(detecciones) == 24, "hay 24 ventanas impares entre 5 y 51"
    assert detecciones["espurias"].max() == 24, \
        "alguna ventana debe recuperar las 24 lecturas espurias"
    mejor = detecciones.loc[detecciones["espurias"].idxmax()]
    print(f"\nmejor ventana {int(mejor['ventana'])}, con "
          f"{int(mejor['espurias'])} espurias y {int(mejor['falsas'])} falsas")
    print("una ventana corta sigue al contaminante y lo esconde, "
          "una ventana larga vuelve a incluir la tendencia de secado")
else:
    print("ejercicio 6 pendiente, complete la celda y ponga REVISAR_6 = True")

## 11. Problemas del capítulo

Se resuelven cuatro problemas de la Sección 2.7 del libro. Los Problemas 2-20
y 2-21 quedaron resueltos como Ejercicios 2 y 6.

### Problema 2-6

Un archivo contiene la fila `2026-04-12 07;30;00;12,4;1,03`. Explique qué
produce leerla con separador de punto y coma y coma decimal.

In [ ]:
import io

fila = "fecha_hora;caudal_l_s;presion_bar\n2026-04-12 07;30;00;12,4;1,03\n"
try:
    pd.read_csv(io.StringIO(fila), sep=";", decimal=",")
except Exception as error:
    print(type(error).__name__, "->", error)

leida = pd.read_csv(io.StringIO(fila), sep=";", decimal=",", header=None,
                    skiprows=1)
print("\ncampos obtenidos:", list(leida.iloc[0]))
print("\nla marca de tiempo trae el punto y coma como separador de la hora, "
      "de modo que la fila se parte en cinco campos y no en tres. El "
      "encabezado declara tres columnas y la lectura falla o desplaza los "
      "datos. La corrección está en el guion de exportación del registrador, "
      "no en el archivo crudo, que no se toca")

### Problema 2-7

Una serie de nivel tiene 18 marcas duplicadas en 8760 registros horarios.
Estime el efecto sobre el volumen medio anual si se concentran en el
estiaje.

In [ ]:
n_total, n_dup = 8760, 18
nivel_estiaje, nivel_medio = 1.20, 2.65        # m

media_correcta = nivel_medio
media_con_dup = ((n_total * nivel_medio + n_dup * nivel_estiaje)
                 / (n_total + n_dup))
sesgo_pct = 100.0 * (media_con_dup - media_correcta) / media_correcta

print(f"nivel medio correcto        {media_correcta:.4f} m")
print(f"nivel medio con duplicados  {media_con_dup:.4f} m")
print(f"sesgo {sesgo_pct:+.3f} %, esto es {abs(sesgo_pct) * 87.6:.1f} "
      "milésimas por cada mil registros")
print("\nel efecto es pequeño en la media anual porque 18 de 8760 pesan poco, "
      "pero deja de serlo en cualquier estadístico del estiaje, donde 18 "
      "duplicados sobre unas pocas centenas de registros sí desplazan el "
      "resultado, y sobre todo altera el conteo de horas bajo un umbral")

### Problema 2-8

Sobre 500 lecturas de conductividad se obtienen media de 1.42 dS/m,
desviación estándar de 0.31 dS/m, mediana de 1.38 dS/m y desviación absoluta
mediana de 0.06 dS/m. Calcule la escala robusta y explique la
discrepancia.

In [ ]:
media_c, desv_c, mediana_c, mad_c = 1.42, 0.31, 1.38, 0.06
escala_c = 1.4826 * mad_c
print(f"escala robusta normalizada {escala_c:.4f} dS/m")
print(f"razón entre la desviación estándar y la escala robusta "
      f"{desv_c / escala_c:.2f}")
print(f"banda clásica de tres sigmas  [{media_c - 3 * desv_c:.3f}, "
      f"{media_c + 3 * desv_c:.3f}] dS/m")
print(f"banda robusta de 3.5 escalas  [{mediana_c - 3.5 * escala_c:.3f}, "
      f"{mediana_c + 3.5 * escala_c:.3f}] dS/m")
print("\nla desviación estándar triplica a la escala robusta, lo cual solo "
      "ocurre si un puñado de lecturas extremas la infló. La mediana apenas "
      "difiere de la media, de modo que el grueso de la serie está sano y la "
      "discrepancia proviene de la cola. Con el criterio clásico la banda "
      "llega hasta 2.35 dS/m y no detecta nada, mientras que la robusta "
      "cierra en 1.69 dS/m")

### Problema 2-9

Un promedio diario de evapotranspiración usa 6 de las 48 lecturas esperadas.
Discuta si debe reportarse y proponga un umbral de cobertura.

In [ ]:
esperadas, disponibles = 48, 6
cobertura = disponibles / esperadas
print(f"cobertura {cobertura:.3f}, esto es {100 * cobertura:.1f} por ciento")

# Efecto de muestrear solo seis instantes de un ciclo diario completo.
horas = np.arange(48) * 0.5
et = np.maximum(0.0, 6.8 * np.sin(np.pi * (horas - 6.0) / 12.0))   # mm/d
generador = np.random.default_rng(SEMILLA)
sub = np.sort(generador.choice(48, size=6, replace=False))
print(f"media con las 48 lecturas {et.mean():.4f} mm/d")
print(f"media con las 6 lecturas  {et[sub].mean():.4f} mm/d")
print(f"error relativo {100 * abs(et[sub].mean() - et.mean()) / et.mean():.1f} %")
print("\nno debe reportarse. El umbral que la asignatura adopta es de 0.80, "
      "el mismo del Listado 2.5, porque con un ciclo diario marcado la "
      "cobertura por debajo de esa fracción deja fuera sistemáticamente una "
      "parte del ciclo y el promedio deja de ser un promedio diario. Un "
      "umbral menor solo se justifica si se demuestra que las lecturas "
      "faltan de manera completamente aleatoria dentro del día")

## Cierre

### Lista de comprobación

Al cerrar el cuaderno el estudiante debe poder hacer lo siguiente sin
consultar la solución.

- Leer un archivo de instrumentación declarando separador, coma decimal, codificación, tipos y marca de tiempo.
- Producir el informe de las verificaciones de la Tabla 2.4 y explicar qué atrapa cada una.
- Reproducir las medias del Ejemplo 2.1 y explicar el sesgo del 18.6 por ciento.
- Aplicar el criterio robusto de la Ecuación 2.1 con y sin la tendencia, y justificar la diferencia entre las dos escalas.
- Agregar una serie invalidando los intervalos cuya cobertura no alcanza el umbral declarado.

### Qué revisar en el libro si algo no salió

- Si la lectura no salió, el Listado 2.2 y los tres comentarios que lo siguen en la Sección 2.2.
- Si el informe de verificación no salió, la Tabla 2.4 y el Listado 2.3.
- Si los atípicos no salieron, la Definición 2.4, la Ecuación 2.1 y el Algoritmo 2.2.
- Si la discusión del punto de ruptura no salió, la Definición 2.5 y el Teorema 2.1.
- Si la agregación no salió, el Listado 2.5 y el párrafo sobre cobertura que lo precede.

### Declaración del uso de asistentes de programación

Este cuaderno se preparó con apoyo de un asistente automático de programación.
Todo fragmento se sometió al protocolo del Algoritmo 2.3 del libro y cada
resultado numérico se comprueba contra la cifra publicada mediante la función
`contra_libro`. La regla de la asignatura es que el ingeniero responde por el
resultado que firma, con independencia de quién haya tecleado las líneas.